# Deep Past Challenge — ByT5 Akkadian→English Translation

Submission notebook for the [Deep Past Initiative: Machine Translation](https://www.kaggle.com/competitions/deep-past-initiative-machine-translation) competition.

**Approach**: Fine-tuned `google/byt5-small` (300M params) on preprocessed training data.

**Constraints**: Kaggle GPU (16GB VRAM), no internet, ≤9 hours runtime.

In [ ]:
import gc
import os
import re
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from tqdm import tqdm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Preprocessing

In [ ]:
# ── Preprocessing (inline — must stay in sync with src/preprocess.py) ─────────

SUBSCRIPT_MAP = str.maketrans('₀₁₂₃₄₅₆₇₈₉ₓ', '0123456789x')
ACCENT_MAP = {'á': 'a2', 'à': 'a3', 'é': 'e2', 'è': 'e3',
              'í': 'i2', 'ì': 'i3', 'ú': 'u2', 'ù': 'u3'}
H_MAP = {'Ḫ': 'H', 'ḫ': 'h'}


def normalize_h(text):
    for src, dst in H_MAP.items():
        text = text.replace(src, dst)
    return text


def normalize_accents(text):
    for src, dst in ACCENT_MAP.items():
        text = text.replace(src, dst)
    return text


def normalize_subscripts(text):
    return text.translate(SUBSCRIPT_MAP)


def normalize_gaps(text):
    text = re.sub(r'\[\s*…\s*…?\s*\]', ' <big_gap> ', text)
    text = re.sub(r'\[x+\]', ' <gap> ', text)
    text = text.replace('…', ' <big_gap> ')
    text = re.sub(r'\bx{2,}\b', ' <gap> ', text)
    return text


def remove_scribal_notations(text):
    text = re.sub(r'<<.*?>>', '', text)
    text = re.sub(r'<([^<>]*)>', r'\1', text)
    text = text.replace('˹', '').replace('˺', '')
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    text = re.sub(r'(?<!\d)[!?]', '', text)
    text = re.sub(r'(?<!\d)/(?!\d)', '', text)
    text = re.sub(r'(?<!\d):(?!\d)', ' ', text)
    return text


def clean_transliteration(text):
    if not isinstance(text, str):
        return ''
    text = normalize_gaps(text)
    text = remove_scribal_notations(text)
    text = normalize_h(text)
    text = normalize_accents(text)
    text = normalize_subscripts(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def postprocess_prediction(text):
    if not isinstance(text, str):
        return ''
    text = normalize_h(text)
    text = normalize_subscripts(text)
    text = remove_scribal_notations(text)
    text = re.sub(r'\b(\w+)(\s+\1)+\b', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print('Preprocessing functions ready.')

## Name Normalization

In [ ]:
# ── Name Normalization (inline — must stay in sync with src/names.py) ──────────
# Uses onomasticon.csv (bundled in model dataset) + OA_Lexicon_eBL.csv (competition data)
# to correct proper noun spellings in model output.

from difflib import SequenceMatcher

# Paths on Kaggle
ONOMASTICON_PATH = '/kaggle/input/byt5-akkadian-finetuned/onomasticon.csv'
LEXICON_PATH = '/kaggle/input/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv'

# Normalization helpers for matching
_ACCENT_MAP_MATCH = {
    'á': 'a', 'à': 'a', 'é': 'e', 'è': 'e',
    'í': 'i', 'ì': 'i', 'ú': 'u', 'ù': 'u',
}
_H_MAP_MATCH = {'Ḫ': 'H', 'ḫ': 'h'}


def _normalize_spelling(text):
    t = text
    for src, dst in _H_MAP_MATCH.items():
        t = t.replace(src, dst)
    for src, dst in _ACCENT_MAP_MATCH.items():
        t = t.replace(src, dst)
    t = t.translate(SUBSCRIPT_MAP)
    t = re.sub(r'[{}()\[\]]', '', t)
    return t.lower().strip()


def _normalize_english_name(name):
    t = name
    for src, dst in _H_MAP_MATCH.items():
        t = t.replace(src, dst)
    for src, dst in _ACCENT_MAP_MATCH.items():
        t = t.replace(src, dst)
    t = t.replace('ā', 'a').replace('ē', 'e').replace('ī', 'i').replace('ū', 'u')
    t = t.replace('ṣ', 's').replace('ṭ', 't').replace('š', 'sh').replace('ṯ', 'th')
    return t.lower().strip()


class NameNormalizer:
    def __init__(self, onomasticon_path=ONOMASTICON_PATH, lexicon_path=LEXICON_PATH):
        self.spelling_to_names = {}
        self.canonical_names = set()
        self._norm_to_canonical = {}

        # Load onomasticon
        if os.path.exists(onomasticon_path):
            ono = pd.read_csv(onomasticon_path)
            for _, row in ono.iterrows():
                name = str(row.get('Name', '')).strip()
                if not name or name == 'nan':
                    continue
                if str(row.get('Duplicate', '')).strip().lower() == 'true':
                    continue
                self.canonical_names.add(name)
                spellings_str = str(row.get('Spellings_semicolon_separated', ''))
                if spellings_str == 'nan':
                    continue
                for sp in spellings_str.split(';'):
                    sp = sp.strip()
                    if sp:
                        norm_sp = _normalize_spelling(sp)
                        if norm_sp not in self.spelling_to_names:
                            self.spelling_to_names[norm_sp] = []
                        if name not in self.spelling_to_names[norm_sp]:
                            self.spelling_to_names[norm_sp].append(name)
            print(f'Onomasticon loaded: {len(self.canonical_names)} names')
        else:
            print(f'WARNING: Onomasticon not found at {onomasticon_path}')

        # Load lexicon PN/GN
        if os.path.exists(lexicon_path):
            lex = pd.read_csv(lexicon_path)
            pn_gn = lex[lex['type'].isin(['PN', 'GN'])]
            for _, row in pn_gn.iterrows():
                form = str(row.get('form', '')).strip()
                norm = str(row.get('norm', '')).strip()
                if form == 'nan' or norm == 'nan' or not form or not norm:
                    continue
                self.canonical_names.add(norm)
                norm_form = _normalize_spelling(form)
                if norm_form not in self.spelling_to_names:
                    self.spelling_to_names[norm_form] = []
                if norm not in self.spelling_to_names[norm_form]:
                    self.spelling_to_names[norm_form].append(norm)
            print(f'Lexicon loaded: total {len(self.spelling_to_names)} spelling entries')
        else:
            print(f'WARNING: Lexicon not found at {lexicon_path}')

        # Build reverse index
        for name in self.canonical_names:
            norm = _normalize_english_name(name)
            if norm not in self._norm_to_canonical or len(name) >= len(self._norm_to_canonical[norm]):
                self._norm_to_canonical[norm] = name

        print(f'NameNormalizer ready: {len(self.spelling_to_names)} spelling entries, {len(self.canonical_names)} canonical names')

    def find_names_in_transliteration(self, transliteration):
        expected = set()
        for word in transliteration.split():
            norm = _normalize_spelling(word)
            names = self.spelling_to_names.get(norm, [])
            expected.update(names)
        return expected

    def normalize_names(self, transliteration, prediction, min_similarity=0.6):
        if not transliteration or not prediction:
            return prediction

        expected_names = self.find_names_in_transliteration(transliteration)
        if not expected_names:
            return prediction

        name_pattern = re.compile(
            r'\b([A-ZÀ-ÖØ-Þ][a-zà-öø-ÿāēīūṣṭšḫ]*(?:-[A-ZÀ-ÖØ-Þa-zà-öø-ÿāēīūṣṭšḫ]+)*)\b'
        )

        skip_words = {
            'The', 'This', 'That', 'These', 'Those', 'His', 'Her', 'Its',
            'He', 'She', 'They', 'We', 'You', 'My', 'Your', 'Our', 'Their',
            'One', 'Two', 'Three', 'Four', 'Five', 'Six', 'Seven', 'Eight',
            'Nine', 'Ten', 'Silver', 'Gold', 'Copper', 'Bronze', 'Iron',
            'Seal', 'Seals', 'Tablet', 'Month', 'Year', 'Day', 'Mina',
            'Minas', 'Shekel', 'Shekels', 'Talent', 'Son', 'Daughter',
            'House', 'City', 'God', 'King', 'Witnesses', 'Witness',
            'From', 'Before', 'After', 'Until', 'Into', 'With', 'About',
            'Concerning', 'Regarding', 'According', 'Says', 'Said',
            'Reckoned', 'Total', 'If', 'When', 'Then', 'Broken', 'Gap',
            'Period', 'Eponymate', 'Eponymy',
        }

        result = prediction
        replacements = []

        for match in name_pattern.finditer(prediction):
            token = match.group(1)
            if token in skip_words or len(token) < 3:
                continue

            token_norm = _normalize_english_name(token)
            best_canonical = None
            best_score = 0.0

            for cand in expected_names:
                cand_norm = _normalize_english_name(cand)
                score = SequenceMatcher(None, token_norm, cand_norm).ratio()
                if score > best_score:
                    best_score = score
                    best_canonical = cand

            if best_canonical and best_score >= min_similarity and best_canonical != token:
                replacements.append((match.start(), match.end(), token, best_canonical, best_score))

        for start, end, old, new, score in sorted(replacements, key=lambda x: x[0], reverse=True):
            result = result[:start] + new + result[end:]

        return result


name_normalizer = NameNormalizer()
print('Name normalization ready.')

## Load Model

In [ ]:
# ── Load fine-tuned model ─────────────────────────────────────────────
# Model uploaded as Kaggle Dataset — update dataset_sources in kernel-metadata.json
MODEL_PATH = '/kaggle/input/byt5-akkadian-finetuned'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# local_files_only=True prevents HF from treating the path as a repo ID
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH, torch_dtype=torch.float32, local_files_only=True)
model = model.to(device)
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')
if device.type == 'cuda':
    print(f'GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

## Load & Preprocess Test Data

In [ ]:
# ── Load test data ───────────────────────────────────────────────────
test_df = pd.read_csv('/kaggle/input/deep-past-initiative-machine-translation/test.csv')
print(f'Test samples: {len(test_df)}')
print(f'Columns: {list(test_df.columns)}')
print(test_df.head())

# Use raw transliterations (model trained on raw text, no preprocessing)
# Preprocessing creates train/test mismatch — baseline approach is raw text
PREFIX = 'translate Akkadian to English: '
texts = [PREFIX + str(t) for t in test_df['transliteration'].tolist()]
print(f'\nPrepared {len(texts)} inputs for inference')

## Generate Translations

In [ ]:
# ── Inference (tuned for 16GB Kaggle GPU) ──────────────────────────────
# ByT5-small is ~1.2GB in fp32. With beam search (4 beams) and 512-length
# byte sequences, peak memory ~6-8GB. Batch size 8 keeps us safely under 16GB.
BATCH_SIZE = 8          # Larger batch for faster inference (baseline uses 16)
NUM_BEAMS = 4           # 4 beams optimal (sweep: 4 > 6 > 8 > 12)
MAX_NEW_TOKENS = 512    # Test data is sentence-level, shorter than training docs
MAX_INPUT_LENGTH = 512  # Must match training max_length
LENGTH_PENALTY = 2.0    # Tuned via sweep: 2.0 >> 1.0 (+3.7 GeoMean on val)
REPETITION_PENALTY = 1.3  # Prevents beam search degeneration (looping). Safe for ByT5 byte-level output.

predictions = []
start = time.time()

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Translating'):
    batch_texts = texts[i:i + BATCH_SIZE]
    inputs = tokenizer(
        batch_texts,
        max_length=MAX_INPUT_LENGTH,
        padding=True,
        truncation=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            length_penalty=LENGTH_PENALTY,
            repetition_penalty=REPETITION_PENALTY,
            early_stopping=True,
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(decoded)

    # Free intermediate tensors to keep memory stable
    del inputs, outputs
    if device.type == 'cuda' and i % 50 == 0:
        torch.cuda.empty_cache()

elapsed = time.time() - start
print(f'\nGenerated {len(predictions)} translations in {elapsed:.1f}s ({elapsed/max(len(texts),1):.2f}s/sample)')
if device.type == 'cuda':
    print(f'Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')


## Post-process & Write Submission

In [ ]:
# ── Post-process & write submission ───────────────────────────────────
# Minimal post-processing — model trained on raw text
predictions = [p.strip() if isinstance(p, str) else '' for p in predictions]

# Ensure no empty predictions (Kaggle rejects empty strings)
predictions = [p if p.strip() else 'broken text' for p in predictions]

submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': predictions,
})

# Validate submission format
assert list(submission.columns) == ['id', 'translation'], f"Bad columns: {list(submission.columns)}"
assert len(submission) == len(test_df), f"Row count mismatch: {len(submission)} vs {len(test_df)}"
assert submission['translation'].isna().sum() == 0, "Found NaN translations"

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Submission saved: {len(submission)} rows')
print(f'Output file: /kaggle/working/submission.csv')
print(f'\nSubmission head:')
print(submission.head(10))

# Show examples
for i in range(min(5, len(predictions))):
    print(f'\n--- Sample {i} ---')
    print(f'Input:  {test_df.iloc[i]["transliteration"][:150]}')
    print(f'Output: {predictions[i][:150]}')